<center><h1>Guo_Haosen_HW3</h1></center>
<br>
<br>

Name: Haosen Guo
<br>
Github Username: iamevan-g
<br>
USC ID: 7183115772

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [3]:
from pathlib import Path
import re
import pandas as pd
import numpy as np

Get the AReM Data Set

In [4]:
data_dir = Path("../data")
records = []

for folder in data_dir.iterdir():
    if folder.is_dir():
        for file in folder.iterdir():
            match = re.search(r"dataset(\d+)", file.stem, re.I)
            if match:
                records.append({
                    "activity": folder.name,
                    "dataset": int(match.group(1)),
                    "file_path": file
                })

arem_files = pd.DataFrame(records)

### (b) Test and Train Data

In [5]:
arem_files["split"] = arem_files.apply(
    lambda row: "test" if row["dataset"] <= (
        2 if row["activity"].lower() in {"bending1", "bending2"} else 3
    ) else "train",
    axis=1
)

test_files = arem_files[arem_files["split"] == "test"].reset_index(drop=True)
train_files = arem_files[arem_files["split"] == "train"].reset_index(drop=True)

print("Total:", len(arem_files))
print("Train:", len(train_files))
print("Test:", len(test_files))

Total: 88
Train: 69
Test: 19


### (c) Feature Extraction

#### i. Research

Common time domain features used in time-series classification include minimum, maximum, range, mean, median, variance, standard deviation, quartiles, interquartile range, root mean square (RMS), skewness, kurtosis, and zero-crossing rate.

#### ii. Extraction

In [6]:
stats = ["min", "max", "mean", "median", "std", "1st quart", "3rd quart"]
rows = []

arem_files = arem_files.sort_values(["activity", "dataset"]).reset_index(drop=True)

for path in arem_files["file_path"]:
    data = pd.read_csv(path, header=None, comment="#",
                       sep=r"[,\s]+", engine="python", names=range(8))
    data = data.iloc[:, 1:7]

    row = {}
    for i, column in enumerate(data.columns, start=1):
        values = data[column]
        features = [
            values.min(), values.max(), values.mean(), values.median(),
            values.std(), values.quantile(0.25), values.quantile(0.75)
        ]
        row.update({f"{stat}{i}": value for stat, value in zip(stats, features)})
    rows.append(row)

feature_data = pd.DataFrame(rows, index=pd.RangeIndex(1, len(rows) + 1, name="Instance"))
feature_data

,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,mean2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
Instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


#### iii. Standard Deviation

In [7]:
rng = np.random.default_rng(552)
n_boot = 1000
values = feature_data.to_numpy()
n = len(values)

boot_std = np.array([
    values[rng.integers(0, n, n)].std(axis=0, ddof=1)
    for _ in range(n_boot)
])

std_ci = pd.DataFrame({
    "std": feature_data.std(),
    "90% CI lower": np.percentile(boot_std, 5, axis=0),
    "90% CI upper": np.percentile(boot_std, 95, axis=0)
})
std_ci

,std,90% CI lower,90% CI upper
min1,9.569975,8.235282,10.689237
max1,4.394362,3.332624,5.304701
mean1,5.335718,4.704834,5.886290
median1,5.440054,4.792167,6.009330
std1,1.772153,1.572041,1.943805
1st quart1,6.153590,5.567811,6.634061
3rd quart1,5.138925,4.329941,5.861211
min2,0.000000,0.000000,0.000000
max2,5.062729,4.639960,5.391927
mean2,1.574164,1.392690,1.704373


I calculated the standard deviation of each feature across the 88 instances. I then generated 10,000 bootstrap samples with replacement and used the 5th and 95th percentiles of the bootstrap standard deviations to form a 90% confidence interval.

#### iv. Select Features

In [8]:
from sklearn.feature_selection import f_classif

train = (arem_files["split"] == "train").to_numpy()
f_scores, _ = f_classif(feature_data[train], arem_files["activity"][train])
f_scores = pd.Series(f_scores, index=feature_data.columns).fillna(0)
f_scores.groupby(f_scores.index.str[:-1]).mean().sort_values(ascending=False)


d:\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:112: UserWarning: Features [ 7 21] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
d:\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


mean         368.170192
3rd quart    350.436027
median       317.745043
1st quart    154.114553
std          109.167132
max           30.030112
min            9.759022
dtype: float64

I select mean, standard deviation, and 3rd quartile. Based on the Anova F-scores on the training set, mean and 3rd quartile best separate the activities, while min and max are the weakest. Since mean, median, and quartiles are highly correlated, I add standard deviation to capture how much the signal fluctuates, which helps distinguish dynamic activities from static ones.

## 2. ISLR 3.7.4

### (a) Linear Train

We would expect the training RSS of the cubic regression to be lower. The cubic model contains the linear model as a special case (β₂ = β₃ = 0), so least squares can always fit the training data at least as well, and the extra terms will fit some of the noise.

### (b) Linear Test

We would expect the test RSS of the linear regression to be lower. Since the true relationship is linear, the extra cubic terms only add variance by overfitting the noise, without reducing bias.

### (c) Not Linear Train

We would expect the training RSS of the cubic regression to be lower. As in (a), the cubic model is more flexible and nests the linear model, so its training RSS is always less than or equal to that of the linear model, regardless of the true relationship.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower test RSS. If the true relationship is only slightly nonlinear, the linear model may generalize better. If the relationship has a stronger nonlinear pattern that a cubic model captures, the cubic model may have a lower test RSS.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 